In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

In [13]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator

In [5]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile"
)

In [6]:
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001FB81DF0BD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001FB81F459D0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [7]:
class EvaluationSchema(BaseModel):
    
    feedback: str = Field(description='Detailed feedback of essay')
    score: int = Field(description='score out of 10', ge=0, le=10)

In [8]:
structured_model = llm.with_structured_output(EvaluationSchema)

In [40]:
essay = """
The Thing About School and Stuff

School is something that people go because everybody says to go there and if not then maybe future become not good and also parents become angry many times. In school there are teachers and students and chairs and lots of homework which honestly nobody likes but still it exists for no reason maybe. The school starts in morning which is very difficult because waking up early is pain and sleep is more better than mathematics.

Teachers always talking about studies and exams and future jobs and success but sometimes students not listening because mind is somewhere else like food or mobile phones or cricket or literally anything except the class. Also homework is given everyday continuously without stopping and then teachers ask why homework not complete even though there are many other things in life too.

Friends in school are one of the thing which makes school less boring otherwise probably everybody would escape from there. Sometimes people fight also over small stupid reasons like pen or bench place or who copied whose answer. Lunch break is best period because finally everybody become alive and starts eating different foods and asking others for one bite even after bringing own lunch.

Exams are horrible disaster moments where students suddenly remember whole year was wasted and now one night before exam everybody trying to study entire books which is impossible thing but still people try. During exam some students become very serious like scientists while others just stare at paper hoping answers appear magically from universe.

In conclusion school is mixture of many random things happening together all time and somehow people survive it and later say it was best days even though at that time everybody was suffering mostly. So school is important maybe or maybe not but definitely it exists and cannot be escaped easily.

"""

In [41]:
prompt = f"Evaluate the language quality of the folllowing essay and provide a feedback and assign a score out of 10 \n {essay}"
output = structured_model.invoke(prompt)

In [42]:
print(output)

feedback="The essay has a clear and relatable topic, but the language quality is poor due to simplicity and informality of the writing style. The text is filled with colloquial expressions and slang, such as 'and stuff', 'literally anything', and 'become alive', which are not suitable for formal writing. Additionally, the essay lacks coherence and organization, with ideas and paragraphs not being clearly connected. The writing also contains grammatical errors, such as missing articles, incorrect verb tenses, and run-on sentences. However, the essay does show some creativity and humor, and the author's voice and perspective are evident throughout. To improve, the author should work on using more formal language, organizing their ideas, and editing for grammar and punctuation errors." score=4


In [43]:
class UPSCState(TypedDict):
    
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    
    individual_score: Annotated[list[int], operator.add]
    avg_score: float

In [44]:
def evaluate_language(state: UPSCState):
    essay = state['essay']
    
    prompt = f"Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}"
    
    output = structured_model.invoke(prompt)
    
    return {'language_feedback': output.feedback, 'individual_score': [output.score]}

In [45]:
def evaluate_analysis(state: UPSCState):
    essay = state['essay']
    
    prompt = f"Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {essay}"
    
    output = structured_model.invoke(prompt)
    
    return {'analysis_feedback': output.feedback, 'individual_score': [output.score]}

In [46]:
def evaluate_thought(state: UPSCState):
    essay = state['essay']
    
    prompt = f"Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {essay}"
    
    output = structured_model.invoke(prompt)
    
    return {'clarity_feedback': output.feedback, 'individual_score': [output.score]}

In [47]:
def final_evaluation(state: UPSCState):
    
    #summary feedback
    prompt = f"Based on the following feedback create a summarized feedback \n language feedback -{state['language_feedback']} \n depth of analysis - {state['analysis_feedback']} \n clarity of thought - {state['clarity_feedback']}"
    final_feedback = llm.invoke(prompt).content
    
    #avg calculate
    avg_score = sum(state['individual_score'])/len(state['individual_score'])
    
    return {'overall_feedback': final_feedback, 'avg_score': avg_score}

In [48]:
graph = StateGraph(UPSCState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)


#create edges

graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation', END)


workflow = graph.compile()


In [49]:
intial_state = {'essay': essay}
final_state = workflow.invoke(intial_state)

In [50]:
print(final_state['language_feedback'])

The essay lacks coherence and organization, with ideas and sentences seemingly thrown together randomly. The language is simplistic and often awkward, with numerous grammatical errors and awkward phrasing. However, the essay does capture the spirit of a student's perspective on school, with some relatable and humorous moments. To improve, the writer should focus on developing a clear thesis statement, organizing their ideas, and refining their language skills.


In [51]:
print(final_state['avg_score'])

5.333333333333333


In [52]:
print(final_state['individual_score'])

[6, 4, 6]


In [39]:
print(final_state['overall_feedback'])

**Summarized Feedback**

The essay is well-structured and effectively conveys the importance of time management for students. It provides a clear analysis of the topic, highlighting its benefits and supporting arguments with relevant examples. However, there are areas for improvement:

* Use more varied vocabulary and sentence structures to make the text more engaging
* Avoid repetitive language and concise paragraphs
* Provide deeper and more nuanced analysis, considering potential counterarguments and complexities
* Refine argumentation and expression with more critical thinking and sophistication
* Include more concrete examples and supporting evidence to strengthen arguments
* Improve sentence structure and clarity for a more direct and concise discussion

Overall, the essay demonstrates a good understanding of the topic, but with refinement and attention to these areas, it can become even more effective in conveying the importance of time management for students.
